<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week18/day3-4/Copy_of_agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [1]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 14.9 MB/s eta 0:00:00


## 1) Define KB

In [2]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [3]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    #To-Do: you can customize the name and description of the tool here for example:
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {
        "query": {
            "type": "string",
            "description": "Keywords or a question to search the knowledge base for.",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "First number."},
        "b": {"type": "number", "description": "Second number."},
        "op": {"type": "string", "description": "'add' or 'multiply'.", "nullable": True},
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()


## 3) Model (tiny local)

In [5]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
    model_id=MODEL_ID,
    max_new_tokens=256,
)

print("Model ready:", MODEL_ID)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [6]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI that uses tools to answer questions. "
        "For math questions, use math_tool. For conceptual questions, use kb_lookup_tool. "
        "Keep answers to 2-4 sentences. When you use the knowledge base, cite the source "
        "tag like [kb:agentic]. If there is no evidence, say so and suggest a follow-up question."
    ),
)

print(agent)

## 5) Test queries

In [7]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': {'type': 'number', 'description': '12'}, 'b': {'type':          │
│ 'number', 'description': '30'}}                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Step 2: Duration 4.56 seconds| Input tokens: 2,757 | Output tokens: 324]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 4.91 seconds| Input tokens: 3,322 | Output tokens: 449]

Answer: An agentic AI loop is a type of AI task that involves using tools to solve a problem repeatedly until a specific outcome is achieved. The loop is designed to be self-repeating, meaning that the AI will keep performing the same action until a desired outcome is achieved. The outcome can be any desired result, such as finding a solution to a problem, generating a piece of text, or providing a final answer to a task. The agentic AI loop is a powerful tool for solving complex problems, as it allows the AI to repeat the same action until a desired outcome is achieved.
